# Intercoder Reliability Analysis

This notebook evaluates the consistency of the manual coding procedure by comparing the researcher's classifications with independent classifications generated by ChatGPT (GPT-6). A random sample of approximately 20% of the dataset for each company is independently classified using the same predefined coding framework. Agreement is assessed separately for Leadership_context and Information_valence using Cohen's kappa (Cohen, 1960) and exact percentage agreement. This analysis provides a reliability assessment of the coding procedure and supports the methodological transparency of the study.

## 1. Dataset Validation

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

# Reproducibility
RANDOM_SEED = 42
SAMPLE_FRACTION = 0.20

# Input files
files = {
    "adidas": "../data/input/preparation/adidas.xlsx",
    "puma": "../data/input/preparation/puma.xlsx",
    "asics": "../data/input/preparation/asics.xlsx",
    "mizuno": "../data/input/preparation/mizuno.xlsx",
}

# Required columns
required_columns = [
    "ID",
    "Text",
    "Text_type",
    "Leadership_context_human",
    "Information_valence_human",
    "Communication_type",
    "Source_type",
    "Company",
    "Country",
    "Year",
    "Covid",
    "Communicator_role",
    "Communicator_name"
]

# Output directory
output_dir_report = Path("../data/input/preparation/quality_reports")
output_dir_report.mkdir(parents=True, exist_ok=True)

# Process each company
for company, file_path in files.items():

    df = pd.read_excel(file_path)

    output_file = output_dir_report / f"{company}_quality_report.txt"

    with open(output_file, "w", encoding="utf-8") as f:

        def log(message=""):
            print(message, file=f)

        log("=" * 60)
        log(f"DATA QUALITY REPORT: {company.upper()}")
        log("=" * 60)

        # 1. Dataset overview
        log("\n1. DATASET OVERVIEW")
        log(f"Dataset size: {df.shape}")
        log(f"Number of rows: {len(df)}")
        log(f"Number of columns: {len(df.columns)}")

        log("\nColumns:")
        for col in df.columns:
            log(f"- {col}")

        # 2. Check required columns
        log("\n2. REQUIRED COLUMNS CHECK")

        missing_columns = [
            col for col in required_columns
            if col not in df.columns
        ]

        if missing_columns:
            log(f"Missing columns: {missing_columns}")
        else:
            log("All required columns are present.")

        # 3. Missing values
        log("\n3. MISSING VALUES")

        for col in required_columns:

            if col not in df.columns:
                continue

            missing = df[col].isna().sum()
            percentage = df[col].isna().mean() * 100

            log(
                f"{col}: {missing} missing values "
                f"({percentage:.2f}%)"
            )

        # 4. Unique values
        log("\n4. UNIQUE VALUES")

        for col in required_columns:

            if col not in df.columns:
                continue

            unique_count = df[col].nunique(dropna=True)

            log(f"{col}: {unique_count} unique values")

        # 5. Category distributions
        log("\n5. CATEGORY DISTRIBUTIONS")

        for col in required_columns:

            if col not in df.columns:
                continue

            # Skip free-text and identifier columns
            if col in ["ID", "Text"]:
                continue

            log(f"\n{col}:")
            log(df[col].value_counts(dropna=False).to_string())

        # 6. Duplicate checks
        log("\n6. DUPLICATE CHECKS")

        if "ID" in df.columns:
            duplicate_ids = df["ID"].dropna().duplicated().sum()
            log(f"Duplicate IDs: {duplicate_ids}")

        if "Text" in df.columns:
            duplicate_texts = df["Text"].dropna().duplicated().sum()
            log(f"Duplicate texts: {duplicate_texts}")

        log("\n" + "=" * 60)
        log("END OF REPORT")
        log("=" * 60)

    print(f"Report saved: {output_file}")

Report saved: ../data/input/preparation/quality_reports/adidas_quality_report.txt
Report saved: ../data/input/preparation/quality_reports/puma_quality_report.txt
Report saved: ../data/input/preparation/quality_reports/asics_quality_report.txt
Report saved: ../data/input/preparation/quality_reports/mizuno_quality_report.txt


## 2. Random Sampling

In [2]:
company_data = {}

for company, file_path in files.items():

    df = pd.read_excel(file_path)

    # Standardize whitespace in relevant string columns
    for col in [
        "ID",
        "Leadership_context_human",
        "Information_valence_human"
    ]:
        df[col] = df[col].map(
            lambda x: x.strip() if isinstance(x, str) else x
        )

    # Select eligible records
    eligible = df.dropna(
        subset=[
            "ID",
            "Text",
            "Leadership_context_human",
            "Information_valence_human"
        ]
    ).copy()

    eligible = eligible[
        eligible["Text"].astype(str).str.strip().ne("")
    ].copy()

    company_data[company] = eligible

    print(
        f"{company.upper()}: "
        f"{len(df)} original records, "
        f"{len(eligible)} eligible records, "
        f"{len(df) - len(eligible)} excluded records."
    )

ADIDAS: 219 original records, 219 eligible records, 0 excluded records.
PUMA: 105 original records, 105 eligible records, 0 excluded records.
ASICS: 206 original records, 206 eligible records, 0 excluded records.
MIZUNO: 62 original records, 62 eligible records, 0 excluded records.


In [ ]:
""" company_samples = {}
sampling_records = []

for company, df in company_data.items():

    sample = df.sample(
        frac=SAMPLE_FRACTION,
        random_state=RANDOM_SEED
    ).sort_values("ID").reset_index(drop=True)

    company_samples[company] = sample

    sampling_records.append({
        "Company": company,
        "Eligible_records": len(df),
        "Sample_records": len(sample),
        "Sample_percentage": round(
            len(sample) / len(df) * 100, 2
        ) if len(df) else np.nan
    })

sampling_summary = pd.DataFrame(sampling_records)
display(sampling_summary)
 """

' company_samples = {}\nsampling_records = []\n\nfor company, df in company_data.items():\n\n    sample = df.sample(\n        frac=SAMPLE_FRACTION,\n        random_state=RANDOM_SEED\n    ).sort_values("ID").reset_index(drop=True)\n\n    company_samples[company] = sample\n\n    sampling_records.append({\n        "Company": company,\n        "Eligible_records": len(df),\n        "Sample_records": len(sample),\n        "Sample_percentage": round(\n            len(sample) / len(df) * 100, 2\n        ) if len(df) else np.nan\n    })\n\nsampling_summary = pd.DataFrame(sampling_records)\ndisplay(sampling_summary)\n '

## 3. Preparation of the blinded coding sample

In [3]:
""" # Output directory
output_dir_ai = Path("../data/input/preparation/ai_results")
output_dir_ai.mkdir(parents=True, exist_ok=True)

ai_input_columns = [
    "ID",
    "Text",
    "Text_type",
    "Communication_type",
    "Source_type",
    "Company",
    "Country",
    "Year",
    "Covid",
    "Communicator_role",
    "Communicator_name"
]

company_references = {}

for company, sample in company_samples.items():

    safe_name = company.lower()

    # Keep only columns that exist in the dataset
    columns_to_export = [
        col for col in ai_input_columns
        if col in sample.columns
    ]

    ai_input = sample[columns_to_export].copy()

    # Researcher's original classifications
    reference = sample[
        [
            "ID",
            "Leadership_context_human",
            "Information_valence_human"
        ]
    ].copy()

    company_references[company] = reference

    ai_input.to_excel(
        output_dir_ai  / f"AI_{safe_name}.xlsx",
        index=False
    )

    reference.to_excel(
        output_dir_ai  / f"AI_reference_{safe_name}.xlsx",
        index=False
    )

    print(f"{company.upper()}: blinded input exported.") """

' # Output directory\noutput_dir_ai = Path("../data/input/preparation/ai_results")\noutput_dir_ai.mkdir(parents=True, exist_ok=True)\n\nai_input_columns = [\n    "ID",\n    "Text",\n    "Text_type",\n    "Communication_type",\n    "Source_type",\n    "Company",\n    "Country",\n    "Year",\n    "Covid",\n    "Communicator_role",\n    "Communicator_name"\n]\n\ncompany_references = {}\n\nfor company, sample in company_samples.items():\n\n    safe_name = company.lower()\n\n    # Keep only columns that exist in the dataset\n    columns_to_export = [\n        col for col in ai_input_columns\n        if col in sample.columns\n    ]\n\n    ai_input = sample[columns_to_export].copy()\n\n    # Researcher\'s original classifications\n    reference = sample[\n        [\n            "ID",\n            "Leadership_context_human",\n            "Information_valence_human"\n        ]\n    ].copy()\n\n    company_references[company] = reference\n\n    ai_input.to_excel(\n        output_dir_ai  / f"AI_{

## 4. Independent AI Coding

Check the `prompt.txt` and the `response.txt` files in `data/input/preparation/ai_results` 

## 5. Data Matching and Disagreement

In [ ]:
# Output directory
output_dir_ai = Path("../data/input/preparation/ai_results")
output_dir_ai.mkdir(parents=True, exist_ok=True)

ai_results_by_company = {}

company_samples = ["adidas", "puma", "asics", "mizuno"]
company_references = {}

for company in company_samples:

    company_references[company] = pd.read_excel(
        output_dir_ai / f"AI_reference_{company}.xlsx",
        dtype={"ID": "string"}
    )


# --------------------------------------------------
# 2. Load AI classification results
# --------------------------------------------------

ai_results_by_company = {}

for company in company_samples:

    file_path = output_dir_ai / f"AI_{company}_classified.xlsx"

    ai_results_by_company[company] = pd.read_excel(
        file_path,
        dtype={"ID": "string"}
    )


# --------------------------------------------------
# 3. Merge reference and AI classifications
# --------------------------------------------------

comparisons_by_company = {}

for company in company_data:

    reference = company_references[company].copy()
    reference["ID"] = reference["ID"].astype("string")
    reference["_original_order"] = range(len(reference))

    ai_results = ai_results_by_company[company]

    comparison = reference.merge(
        ai_results[
            [
                "ID",
                "Leadership_context_AI",
                "Information_valence_AI"
            ]
        ],
        on="ID",
        how="outer",
        validate="one_to_one",
        indicator=True
    )

    # Check that all observations match
    if not comparison["_merge"].eq("both").all():
        raise ValueError(
            f"{company}: unmatched records found."
        )

    # Restore original reference observation order
    comparison = (
        comparison
        .sort_values("_original_order")
        .drop(columns=["_merge", "_original_order"])
        .reset_index(drop=True)
    )

    # Add company identifier
    comparison["Company"] = company

    # Store comparison
    comparisons_by_company[company] = comparison

    # Export comparison file
    comparison.to_excel(
        output_dir_ai / f"coding_comparison_{company}.xlsx",
        index=False
    )

    print(
        f"{company.upper()}: "
        f"{len(comparison)} classifications matched."
    )

ADIDAS: 44 classifications matched.
PUMA: 21 classifications matched.
ASICS: 41 classifications matched.
MIZUNO: 12 classifications matched.


In [22]:
# Output directory
output_dir_ai = Path("../data/input/preparation/ai_results/quality_reports")
output_dir_ai.mkdir(parents=True, exist_ok=True)


for company, comparison in comparisons_by_company.items():

    for researcher_col, ai_col in [
        ("Leadership_context_human", "Leadership_context_AI"),
        ("Information_valence_human", "Information_valence_AI")
    ]:

        researcher = comparison[researcher_col].astype("string").str.strip()
        ai = comparison[ai_col].astype("string").str.strip()

        valid = (
            researcher.notna()
            & ai.notna()
            & researcher.ne("")
            & ai.ne("")
        )

        disagreements = comparison.loc[
            valid & (researcher != ai)
        ].copy()

        print(
            f"{company.upper()} — {researcher_col}: "
            f"{len(disagreements)} disagreements"
        )

        disagreements.to_excel(
            output_dir_ai
            / f"disagreements_{company}_{researcher_col}.xlsx",
            index=False
        ) 

ADIDAS — Leadership_context_human: 22 disagreements
ADIDAS — Information_valence_human: 44 disagreements
PUMA — Leadership_context_human: 11 disagreements
PUMA — Information_valence_human: 21 disagreements
ASICS — Leadership_context_human: 20 disagreements
ASICS — Information_valence_human: 41 disagreements
MIZUNO — Leadership_context_human: 6 disagreements
MIZUNO — Information_valence_human: 12 disagreements


## 7. Cohen's Kappa 

In [29]:
def normalize_labels(series, allowed_categories):

    values = series.astype("string").str.strip()

    canonical = {
        category.casefold(): category
        for category in allowed_categories
    }

    return (
        values.str.casefold()
        .map(canonical)
        .fillna(values)
    )

def calculate_reliability(
    comparison,
    researcher_col,
    ai_col,
    allowed_categories
):

    # Standardize labels
    researcher = normalize_labels(
        comparison[researcher_col],
        allowed_categories
    )

    ai = normalize_labels(
        comparison[ai_col],
        allowed_categories
    )

    # Identify valid observations
    valid = (
        researcher.notna()
        & ai.notna()
        & researcher.ne("")
        & ai.ne("")
    )

    researcher = researcher[valid].astype(str)
    ai = ai[valid].astype(str)

    if len(researcher) == 0:
        return {
            "N_valid_pairs": 0,
            "Exact_agreement_pct": np.nan,
            "Cohens_kappa": np.nan
        }

    # Validate categories
    observed = set(researcher) | set(ai)

    unexpected = observed - set(allowed_categories)

    if unexpected:
        raise ValueError(
            f"Unexpected coding labels: {sorted(unexpected)}"
        )

    # Calculate exact agreement
    agreement = (
        researcher.to_numpy() == ai.to_numpy()
    ).mean() * 100

    # Calculate Cohen's kappa
    kappa_value = cohen_kappa_score(
        researcher,
        ai
    )

    return {
        "N_valid_pairs": len(researcher),

        "Exact_agreement_pct": round(
            agreement, 2
        ),

        "Cohens_kappa": (
            round(float(kappa_value), 4)
            if np.isfinite(kappa_value)
            else np.nan
        )
    }

In [30]:
# ============================================================
# HUMAN–AI CODING RELIABILITY ANALYSIS
# ============================================================


from sklearn.metrics import confusion_matrix

# Output directory
output_dir_ai = Path(
    "../data/input/preparation/ai_results/quality_reports"
)

output_dir_ai.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Define coding dimensions and valid categories
# ------------------------------------------------------------

coding_dimensions = {
    "Leadership_context": {
        "human": "Leadership_context_human",
        "ai": "Leadership_context_AI",
        "categories": [
            "PERF",
            "STRAT",
            "MARKET",
            "CHALL",
            "PEOPLE",
            "STAKE",
            "OTHER"
        ]
    },

    "Information_valence": {
        "human": "Information_valence_human",
        "ai": "Information_valence_AI",
        "categories": [
            "Positive",
            "Neutral",
            "Negative"
        ]
    }
}


# ------------------------------------------------------------
# 2. Interpret Cohen's kappa
# ------------------------------------------------------------

def interpret_kappa(kappa):

    if pd.isna(kappa):
        return "Undefined"

    elif kappa < 0:
        return "Below chance agreement"

    elif kappa <= 0.20:
        return "Slight agreement"

    elif kappa <= 0.40:
        return "Fair agreement"

    elif kappa <= 0.60:
        return "Moderate agreement"

    elif kappa <= 0.80:
        return "Substantial agreement"

    else:
        return "Almost perfect agreement"


# ------------------------------------------------------------
# 3. Combine company datasets for pooled analysis
# ------------------------------------------------------------

all_comparisons = pd.concat(
    comparisons_by_company.values(),
    ignore_index=True
)

datasets_to_analyze = {
    **comparisons_by_company,
    "ALL_COMPANIES": all_comparisons
}


# ------------------------------------------------------------
# 4. Calculate reliability
# ------------------------------------------------------------

reliability_results = []
analysis_lines = []

for company, comparison in datasets_to_analyze.items():

    print(f"\n{'=' * 60}")
    print(f"COMPANY: {company.upper()}")
    print(f"{'=' * 60}")

    analysis_lines.append(f"\nCOMPANY: {company.upper()}")

    for dimension, config in coding_dimensions.items():

        researcher_col = config["human"]
        ai_col = config["ai"]
        categories = config["categories"]

        # Calculate reliability using your existing function
        metrics = calculate_reliability(
            comparison=comparison,
            researcher_col=researcher_col,
            ai_col=ai_col,
            allowed_categories=categories
        )

        # Clean labels
        researcher = (
            comparison[researcher_col]
            .astype("string")
            .str.strip()
            .fillna("")
        )

        ai = (
            comparison[ai_col]
            .astype("string")
            .str.strip()
            .fillna("")
        )

        valid = (
            researcher.ne("")
            & ai.ne("")
        )

        # Identify disagreements
        disagreement_mask = valid & (researcher != ai)

        disagreements = comparison.loc[
            disagreement_mask
        ].copy()

        # Interpret kappa
        interpretation = interpret_kappa(
            metrics["Cohens_kappa"]
        )

        # Store reliability results
        reliability_results.append({
            "Company": company,
            "Dimension": dimension,
            "N_total": len(comparison),
            "N_valid_pairs": metrics["N_valid_pairs"],
            "N_excluded": (
                len(comparison) - metrics["N_valid_pairs"]
            ),
            "Exact_agreement_pct": metrics["Exact_agreement_pct"],
            "Cohens_kappa": metrics["Cohens_kappa"],
            "N_disagreements": len(disagreements),
            "Interpretation": interpretation
        })

        # Print results
        print(f"\nDimension: {dimension}")

        print(
            f"Valid observations: "
            f"{metrics['N_valid_pairs']}"
        )

        print(
            f"Exact agreement: "
            f"{metrics['Exact_agreement_pct']:.2f}%"
        )

        print(
            f"Cohen's kappa: "
            f"{metrics['Cohens_kappa']:.4f}"
        )

        print(
            f"Interpretation: {interpretation}"
        )

        print(
            f"Disagreements: {len(disagreements)}"
        )

        # Save disagreements
        disagreements.to_excel(
            output_dir_ai
            / f"disagreements_{company}_{dimension}.xlsx",
            index=False
        )

        # Create confusion matrix
        matrix = confusion_matrix(
            researcher[valid],
            ai[valid],
            labels=categories
        )

        confusion_df = pd.DataFrame(
            matrix,
            index=categories,
            columns=categories
        )

        confusion_df.index.name = "Human"
        confusion_df.columns.name = "AI"

        confusion_df.to_excel(
            output_dir_ai
            / f"confusion_{company}_{dimension}.xlsx"
        )

        # Identify most frequent disagreement
        disagreement_pairs = pd.DataFrame({
            "Human": researcher[disagreement_mask].to_numpy(),
            "AI": ai[disagreement_mask].to_numpy()
        })

        if len(disagreement_pairs) > 0:

            pair_counts = (
                disagreement_pairs
                .value_counts()
            )

            most_common_pair = pair_counts.index[0]
            most_common_count = int(pair_counts.iloc[0])

            confusion_description = (
                f"Most frequent disagreement: "
                f"Human={most_common_pair[0]}, "
                f"AI={most_common_pair[1]} "
                f"({most_common_count} observations)."
            )

        else:
            confusion_description = (
                "No disagreements among valid observations."
            )

        # Generate descriptive analysis
        analysis_lines.append(
            f"\n{dimension}\n"
            f"Valid observations: {metrics['N_valid_pairs']}\n"
            f"Exact agreement: {metrics['Exact_agreement_pct']:.2f}%\n"
            f"Cohen's kappa: {metrics['Cohens_kappa']:.4f}\n"
            f"Interpretation: {interpretation}\n"
            f"Disagreements: {len(disagreements)}\n"
            f"{confusion_description}\n"
        )


# ------------------------------------------------------------
# 5. Create summary table
# ------------------------------------------------------------

reliability_summary = pd.DataFrame(
    reliability_results
)

reliability_summary.to_excel(
    output_dir_ai / "human_ai_reliability_summary.xlsx",
    index=False
)


# ------------------------------------------------------------
# 6. Save descriptive analysis
# ------------------------------------------------------------

analysis_text = "\n".join(analysis_lines)

(output_dir_ai / "reliability_analysis.txt").write_text(
    analysis_text,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("FINAL HUMAN–AI RELIABILITY SUMMARY")
print("=" * 60)

display(reliability_summary)


COMPANY: ADIDAS

Dimension: Leadership_context
Valid observations: 44
Exact agreement: 50.00%
Cohen's kappa: 0.4094
Interpretation: Moderate agreement
Disagreements: 22

Dimension: Information_valence
Valid observations: 44
Exact agreement: 65.91%
Cohen's kappa: 0.3726
Interpretation: Fair agreement
Disagreements: 44


ValueError: At least one label specified must be in y_true